# **EASD Race Results - Gather the Data**

Run on Python 3.14 | No errors | No warnings

In [14]:
# Import packages

# For data manipulation and analysis
import pandas as pd

# Standard library: API requests, validation, and retry handling
import json
import re
import time
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

In [ ]:
# Set race parameters
RACE_ID = 186860
RESULT_SET_ID = 661338
YEAR = 2026

In [16]:
# Define function to fetch a single page of race results from RunSignup
def fetch_page(race_id, result_set_id, page, per_page):
    query = urlencode({"resultSetId": result_set_id, "page": page, "num": per_page})
    request = Request(
        f"https://runsignup.com/Race/Results/{race_id}?{query}",
        headers={"X-Requested-With": "XMLHttpRequest", "Accept": "application/json",
                 "User-Agent": "EASD-results-downloader/1.0"},
    )
    for attempt in range(3):
        try:
            with urlopen(request, timeout=45) as response:
                payload = json.load(response)
            break
        except HTTPError as exc:
            if exc.code not in (429, 500, 502, 503, 504) or attempt == 2:
                raise RuntimeError(f"Results request failed: HTTP {exc.code}") from exc
            time.sleep(2 ** (attempt + 1))
        except (URLError, TimeoutError) as exc:
            if attempt == 2:
                raise RuntimeError("Cannot reach RunSignup. Check your internet connection.") from exc
            time.sleep(2 ** (attempt + 1))
        except (ValueError, UnicodeError) as exc:
            raise RuntimeError("RunSignup did not return results JSON; its endpoint may have changed.") from exc
    if not isinstance(payload, dict) or payload.get("error"):
        raise RuntimeError(f"RunSignup returned an error: {payload}")
    if not isinstance(payload.get("headings"), list) or not isinstance(payload.get("resultSet"), dict):
        raise RuntimeError("RunSignup's results format has changed.")
    return payload

In [17]:
# Define function to normalize race result times to a consistent HH:MM:SS format
def normalize_time(value):
    """Normalize clock time to HH:MM:SS; preserve blanks and status values."""
    value = str(value or "").strip()
    match = re.fullmatch(r"(?:(\d+):)?(\d+):(\d{2}(?:\.\d+)?)", value)
    if not match:
        return value
    hours, minutes, seconds = match.groups()
    if int(minutes) >= 60 and hours is not None or float(seconds) >= 60:
        return value
    total_minutes = int(hours or 0) * 60 + int(minutes)
    return f"{total_minutes // 60:02}:{total_minutes % 60:02}:{seconds}"

In [18]:
# Define function to download and normalize race results from RunSignup
def download_results(race_id=RACE_ID, result_set_id=RESULT_SET_ID, per_page=1000):
    """Return (column names, rows), failing if any page is missing or repeated."""
    if per_page < 1 or per_page > 1000:
        raise ValueError("per_page must be between 1 and 1000.")
    all_rows = []
    seen_ids = set()
    seen_pages = set()
    headings = None
    expected_total = None
    page = 1
    while True:
        payload = fetch_page(race_id, result_set_id, page, per_page)
        result_set = payload["resultSet"]
        if str(result_set.get("setInfo", {}).get("individual_result_set_id")) != str(result_set_id):
            raise RuntimeError("RunSignup returned a different result set.")
        total = int(result_set["numResults"])
        if headings is None:
            headings = payload["headings"]
            expected_total = total
        elif payload["headings"] != headings or total != expected_total:
            raise RuntimeError("Results changed during download. Run again to get a consistent snapshot.")
        rows = result_set["results"]
        expected_on_page = min(per_page, max(0, expected_total - len(all_rows)))
        if len(rows) != expected_on_page:
            raise RuntimeError(f"Page {page}: expected {expected_on_page} rows, received {len(rows)}.")
        if any(len(row) != len(headings) for row in rows):
            raise RuntimeError(f"Page {page}: row values do not match the headings.")
        if rows:
            signature = json.dumps(rows, ensure_ascii=False)
            if signature in seen_pages:
                raise RuntimeError(f"Page {page} repeats an earlier page.")
            seen_pages.add(signature)
        ids = result_set.get("resultIds")
        if ids is None:
            ids = payload.get("resultUrls")
        if ids is not None:
            if len(ids) != len(rows):
                raise RuntimeError("Result IDs do not match the row count.")
            for result_id in ids:
                identity = str(result_id)
                if identity in seen_ids:
                    raise RuntimeError(f"Repeated result identity {identity}; download is incomplete.")
                seen_ids.add(identity)
        all_rows.extend(rows)
        print(f"Page {page}: {len(all_rows):,} of {expected_total:,} results")
        if len(all_rows) == expected_total:
            break
        page += 1
        time.sleep(0.25)

    visible = [(i, " ".join(str(h["name"]).split()), h["key"])
               for i, h in enumerate(headings) if h.get("hidden") not in (True, "T")]
    columns = [name for _, name, _ in visible]
    if "Clock Time" not in columns or len(set(columns)) != len(columns):
        raise RuntimeError("Expected a unique set of columns including Clock Time.")
    clean_rows = []
    for row in all_rows:
        clean_rows.append([
            normalize_time(row[i]) if key == "clock_time" else
            "" if row[i] is None else str(row[i])
            for i, _, key in visible
        ])
    return columns, clean_rows

In [19]:
# Download race results
headers, all_results = download_results(
    race_id=RACE_ID, result_set_id=RESULT_SET_ID, per_page=1000
)

Page 1: 1,000 of 3,737 results
Page 2: 2,000 of 3,737 results
Page 3: 3,000 of 3,737 results
Page 4: 3,737 of 3,737 results


In [20]:
# Convert results to DataFrame
df0 = pd.DataFrame(all_results, columns=headers)
print(f"Downloaded {len(df0):,} results")

Downloaded 3,737 results


In [21]:
# Verify the columns and data
df0.head()

,Place,Bib,Name,Country,Gender,Clock Time,Pace,Age Group Place,Age Group
0,1,2421,Mohammed Sohail Ahmed,IN,M,00:14:45,2:57,1,M3539
1,2,2216,BASSIGUE KOUSSEMIA,TD,M,00:14:45,2:57,1,M3034
2,3,3787,Gulshan Darji,IN,M,00:14:45,2:57,2,M3539
3,4,4084,DECHACHE BACHIR,DZ,M,00:14:45,2:57,1,M4549
4,5,8503,Hichem Taib,DZ,M,00:14:45,2:57,3,M3539


In [22]:
# Check the data types and non-null values
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Place            3737 non-null   str  
 1   Bib              3737 non-null   str  
 2   Name             3737 non-null   str  
 3   Country          3737 non-null   str  
 4   Gender           3737 non-null   str  
 5   Clock Time       3737 non-null   str  
 6   Pace             3737 non-null   str  
 7   Age Group Place  3737 non-null   str  
 8   Age Group        3737 non-null   str  
dtypes: str(9)
memory usage: 262.9 KB


In [ ]:
# Rename Clock Time column to Time
df0 = df0.rename(columns={'Clock Time': 'Time'})

In [24]:
# Verify the columns and data
df0.head()

,Place,Bib,Name,Country,Gender,Time,Pace,Age Group Place,Age Group
0,1,2421,Mohammed Sohail Ahmed,IN,M,00:14:45,2:57,1,M3539
1,2,2216,BASSIGUE KOUSSEMIA,TD,M,00:14:45,2:57,1,M3034
2,3,3787,Gulshan Darji,IN,M,00:14:45,2:57,2,M3539
3,4,4084,DECHACHE BACHIR,DZ,M,00:14:45,2:57,1,M4549
4,5,8503,Hichem Taib,DZ,M,00:14:45,2:57,3,M3539


In [25]:
# Check the data types and non-null values
df0.info()

<class 'pandas.DataFrame'>
RangeIndex: 3737 entries, 0 to 3736
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   Place            3737 non-null   str  
 1   Bib              3737 non-null   str  
 2   Name             3737 non-null   str  
 3   Country          3737 non-null   str  
 4   Gender           3737 non-null   str  
 5   Time             3737 non-null   str  
 6   Pace             3737 non-null   str  
 7   Age Group Place  3737 non-null   str  
 8   Age Group        3737 non-null   str  
dtypes: str(9)
memory usage: 262.9 KB


In [26]:
# Save df0 to csv
df0.to_csv("EASD Race Results.csv", index=False)